# TPCA Final Business Challenge — Booking Business Review

Executive answer to *“How is our booking business performing, and where should management focus its attention?”* for the TPCA airline lakehouse (1994–1998). Evidence queries E1–E8 below re-run the key metrics against the Gold layer (`tpca.gold.*`), and the final cell presents the board-level summary: 3 key findings, 2 opportunities, 2 risks, each with its source query, metric value, and a recommended management action. All data is synthetic practice data — no real-world causation is claimed.

In [0]:
-- E1 (re-run of Q01): Overall KPI summary, fact grain (bookings WITH flight segments)
SELECT round(sum(net_revenue), 2) AS net_revenue,
       count(DISTINCT booking_id) AS distinct_bookings,
       count(DISTINCT passenger_id) AS active_passengers,
       round(sum(net_revenue) / count(DISTINCT booking_id), 2) AS avg_booking_value
FROM tpca.gold.fact_bookings

net_revenue,distinct_bookings,active_passengers,avg_booking_value
42382952.76,7621,1493,5561.34


In [0]:
-- E2 (re-run of Q01 semantics on the booking table): ALL bookings incl. those with no flight segments
SELECT count(DISTINCT booking_id)   AS total_bookings,
       count(DISTINCT passenger_id) AS active_passengers
FROM tpca.gold.fact_bookings

total_bookings,active_passengers
7621,1493


In [0]:
-- E3 (re-run of Q03): Annual net revenue with year-over-year change
WITH annual AS (
  SELECT year(booking_date) AS booking_year, sum(net_revenue) AS net_revenue
  FROM tpca.gold.fact_bookings
  GROUP BY 1
)
SELECT booking_year,
       round(net_revenue, 2) AS net_revenue,
       round(100 * (net_revenue / lag(net_revenue) OVER (ORDER BY booking_year) - 1), 2) AS yoy_change_pct
FROM annual
ORDER BY booking_year

booking_year,net_revenue,yoy_change_pct
1994,8563262.39,null
1995,8064298.92,-5.83
1996,8407715.81,4.26
1997,8494925.56,1.04
1998,8852750.07,4.21


In [0]:
-- E4 (re-run of Q14): Revenue and volume split by refund flag
SELECT refund_flag,
       round(sum(net_revenue), 2) AS net_revenue,
       round(100 * sum(net_revenue) / sum(sum(net_revenue)) OVER (), 2) AS revenue_share_pct,
       count(DISTINCT booking_id) AS bookings
FROM tpca.gold.fact_bookings
GROUP BY refund_flag
ORDER BY refund_flag

refund_flag,net_revenue,revenue_share_pct,bookings
A,10537190.90,24.86,4359
N,27655599.81,65.25,6893
R,4190162.05,9.89,2153


In [0]:
-- E5 (re-run of Q15): Average discount by travel purpose (line-level average of discount_rate)
SELECT p.travel_purpose,
       round(100 * avg(f.discount_rate), 2) AS avg_discount_pct,
       count(*)                             AS booking_lines
FROM tpca.gold.fact_bookings f
JOIN tpca.gold.dim_passenger p ON p.passenger_id = f.passenger_id
GROUP BY p.travel_purpose
ORDER BY avg_discount_pct DESC

travel_purpose,avg_discount_pct,booking_lines
LEISURE,7.03,5133
CREW,6.88,4775
MILITARY,6.87,4907
STUDENT,6.86,4875
BUSINESS,6.78,5310


In [0]:
-- E6 (re-run of Q20): Passenger geography vs airline geography revenue by region
WITH passenger_side AS (
  SELECT g.region_name, sum(f.net_revenue) AS passenger_region_revenue
  FROM tpca.gold.fact_bookings f
  JOIN tpca.gold.dim_geography g ON g.nation_key = f.passenger_nation_key
  GROUP BY 1
),
airline_side AS (
  SELECT g.region_name, sum(f.net_revenue) AS airline_region_revenue
  FROM tpca.gold.fact_bookings f
  JOIN tpca.gold.dim_geography g ON g.nation_key = f.airline_nation_key
  GROUP BY 1
)
SELECT coalesce(p.region_name, a.region_name) AS region_name,
       round(coalesce(p.passenger_region_revenue, 0), 2) AS passenger_region_net_revenue,
       round(coalesce(a.airline_region_revenue, 0), 2)   AS airline_region_net_revenue,
       round(coalesce(p.passenger_region_revenue, 0) - coalesce(a.airline_region_revenue, 0), 2)
                                                         AS net_revenue_difference
FROM passenger_side p
FULL OUTER JOIN airline_side a ON a.region_name = p.region_name
ORDER BY net_revenue_difference DESC

region_name,passenger_region_net_revenue,airline_region_net_revenue,net_revenue_difference
EUROPE,9178368.66,8450213.25,728155.42
AFRICA,8653216.83,8419047.56,234169.28
ASIA,8427560.29,8558547.61,-130987.32
MIDDLE EAST,8262995.33,8466555.49,-203560.16
AMERICA,7860811.64,8488588.86,-627777.21


In [0]:
-- E7 (re-run of Q16): Flight duration check (data-quality signal)
SELECT round(avg(datediff(arrival_date, departure_date)), 2) AS avg_duration_days,
       count(*) AS booking_lines
FROM tpca.gold.fact_bookings

avg_duration_days,booking_lines
0.0,25000


In [0]:
-- E8: Silver quarantine volumes (data-quality exposure from the cleansing pipeline)
SELECT 'flight_booking'  AS table_name, count(*) AS quarantined_rows FROM tpca.silver.flight_booking_quarantine
UNION ALL SELECT 'booking',        count(*) FROM tpca.silver.booking_quarantine
UNION ALL SELECT 'passenger',      count(*) FROM tpca.silver.passenger_quarantine
UNION ALL SELECT 'airline_route',  count(*) FROM tpca.silver.airline_route_quarantine
UNION ALL SELECT 'route',          count(*) FROM tpca.silver.route_quarantine
UNION ALL SELECT 'region',         count(*) FROM tpca.silver.region_quarantine
UNION ALL SELECT 'airline',        count(*) FROM tpca.silver.airline_quarantine
UNION ALL SELECT 'nation',         count(*) FROM tpca.silver.nation_quarantine
ORDER BY quarantined_rows DESC

table_name,quarantined_rows
flight_booking,165
booking,45
passenger,18
airline_route,10
route,3
airline,2
region,0
nation,0


# Final Business Challenge — How is our booking business performing, and where should management focus?

Board-level summary of the TPCA booking business, 1994–1998, built from the Gold layer. Every cited metric is labeled with its source query (Q01–Q20 from the *SQL buisness Analysis* notebook); cells E1–E8 above re-ran the key queries in this notebook and reproduce those results. Supporting assets: the Bronze ingestion pipeline, the Silver cleansing notebook, the Gold notebook, the **SkyLedger Airline Executive Analytics** dashboard, and a Genie space used for interactive questioning.

> **Data caveat:** all figures come from synthetic practice data. Findings below are descriptions of that data only — no real-world causation is claimed.

---

## 1. Key findings (directly supported by query results)

### Finding 1 — Revenue is in a sustained recovery; 1998 is the best year on record
Q03 shows annual net revenue of **$8.56M (1994) → $8.06M (1995, −5.8%) → $8.41M (1996, +4.3%) → $8.49M (1997, +1.0%) → $8.85M (1998, +4.2%)** — three consecutive growth years (re-verified in E3).

**Recommended management action:** identify the drivers behind the 1996 and 1998 growth years and set a ≥4% YoY revenue target for 1999.

### Finding 2 — The revenue base is highly diversified: no customer, nation, or segment dependency
Q06 shows the top 10 passengers contribute only **$786.3K = 1.86%** of **$42.38M** total net revenue; Q13 shows all 25 nations contribute just **3.6%–5.3%** each (FRANCE leads at 5.31%, $2.25M); Q04 shows revenue spread evenly across travel purposes (largest is BUSINESS at **$9.08M, 21.4%**). Q02 additionally shows monthly revenue stable at **$0.60–0.86M** with no strong seasonality.

**Recommended management action:** direct loyalty investment at breadth (repeat-purchase incentives across the 1,494-passenger active base) rather than defending a handful of accounts, and monitor concentration quarterly.

### Finding 3 — Long-haul flying is the revenue engine, and the carrier base is uniform
Q09 shows **ULTRA_LONG_HAUL $11.86M (28.0%)** plus **LONG_HAUL $11.35M (26.8%)** — together ≈**55%** of net revenue; Q07 shows the top route **GRU-MAD at $511K** with the top 10 closely bunched ($465K–$511K); Q10 shows all 10 airlines within a **<4% spread ($4.18M–$4.33M)**, so ratings do not translate into revenue.

**Recommended management action:** prioritize wide-body capacity and premium inventory on long-haul routes; base carrier partnership decisions on cost and service quality, not revenue concentration.

---

## 2. Opportunities (data evidence + clearly-labeled hypotheses)

### Opportunity 1 — Discounting is undifferentiated; pricing levers are unused
**Data evidence:** Q15 shows average discounts are nearly flat across purposes — **LEISURE 7.03% vs BUSINESS 6.78%** (re-verified in E5) — and Q17 shows revenue per booking is flat across all five priority levels (**$5,508–$5,632**).
**Hypothesis (not evidenced):** a differentiated discount framework could recover margin on BUSINESS fares; the size of any uplift is an assumption, not a measured result.

**Recommended management action:** pilot a segmented discount framework (advance-purchase / flexibility tiers) and measure the uplift against the Q15/Q17 baselines.

### Opportunity 2 — AMERICA is the under-indexing demand region
**Data evidence:** Q12 shows AMERICA passengers generate **$7.86M** vs EUROPE **$9.18M**; Q20 shows the geography asymmetry — passenger-side revenue exceeds airline-side by **+$728K in EUROPE** while AMERICA is **−$628K** (American carriers out-sell American passengers) (re-verified in E6).
**Hypothesis (not evidenced):** this gap represents addressable demand for growth; that cannot be proven from synthetic data.

**Recommended management action:** target commercial initiatives (local partnerships, fare promotions) at AMERICA and re-measure Q12/Q20 half-yearly.

---

## 3. Risks / areas for further investigation (data evidence)

### Risk 1 — Refund-linked revenue is material, and the flag semantics are unverified
**Data evidence:** Q14 shows R-flagged segments carry **$4.19M (9.9%)** of net revenue and A-flagged segments **$10.54M (24.9%)**, with non-refund (N) at **$27.66M (65.3%)** (re-verified in E4). Roughly a tenth of revenue sits on refund-related segments — and the business meaning of the A/R codes has not been confirmed.

**Recommended management action:** audit the refund process (root causes, fees, policy leakage) and confirm the A/R code definitions; set a control to hold refund-linked revenue below 10% of total.

### Risk 2 — Data-quality signals must be resolved before operational decisions
**Data evidence:** Q16/E7 shows average flight duration is **0.0 days for all 25,000 segments** (arrival_date always equals departure_date — a date-capture anomaly). E8 shows **243 rows quarantined** during silver cleansing (flight_booking 165, booking 45, passenger 18, airline_route 10, route 3, airline 2, region 0, nation 0).

**Recommended management action:** validate the booking-date pipeline (same-day arrivals) and review quarantine/rejection handling before trusting duration, volume, or no-show analyses.

---

## 4. What is data vs what is assumption

* **Directly supported by query results:** Findings 1–3, the evidence lines in Opportunities 1–2 and Risks 1–2 (all re-runnable in cells E1–E8 above).
* **Assumptions / hypotheses (not evidenced):** the revenue uplift estimated in Opportunity 1, the addressability of the AMERICA gap in Opportunity 2, persistence of the 1996–1998 growth trend into 1999, and any root-cause explanation for the refund flags or the same-day-arrival anomaly.
* **Grain note:** Q01/E1 KPIs are fact-grain — **$42,382,952.76 net revenue, 7,621 bookings with flight segments, 1,493 active passengers, $5,561.34 avg booking value**. The booking-table grain (E2) counts **8,040 total bookings** including bookings with no flight segments (avg booking value $5,271.51 at that grain). Cite the grain explicitly when reusing these numbers.